# NVIDIA Build: Tool-calling detection via Playwright (DOM)

This notebook runs Playwright against NVIDIA Build model pages and detects:
- A **Tools** button/tab in the rendered DOM (and whether it's disabled)
- A **wrench** icon in the rendered DOM

Targets:
- https://build.nvidia.com/openai/gpt-oss-120b
- https://build.nvidia.com/deepseek-ai/deepseek-r1
- https://build.nvidia.com/deepseek-ai/deepseek-r1-0528

> Requires **internet access** on the machine running the notebook and Playwright browser binaries.


In [1]:
# Install deps (run once)
# %pip -q install playwright


In [2]:
# Install Chromium for Playwright (run once)
# If you're on Linux and hit missing deps, also run:
# !playwright install-deps
# !playwright install chromium


In [3]:
import re
from typing import Dict, Any, List

from playwright.async_api import async_playwright, TimeoutError as PWTimeout

URLS = [
    "https://build.nvidia.com/openai/gpt-oss-120b",
    "https://build.nvidia.com/deepseek-ai/deepseek-r1",
    "https://build.nvidia.com/deepseek-ai/deepseek-r1-0528",
]

async def detect_one(page, url: str) -> Dict[str, Any]:
    out: Dict[str, Any] = {
        "url": url,
        "tools_button_found": False,
        "tools_button_disabled": None,
        "tools_button_outer_html": None,
        "wrench_found": False,
        "wrench_count": 0,
    }

    await page.goto(url, wait_until="domcontentloaded", timeout=60000)
    try:
        await page.wait_for_load_state("networkidle", timeout=20000)
    except PWTimeout:
        pass

    # Find "Tools" button/tab
    tools_btn = page.get_by_role("button", name=re.compile(r"^Tools$", re.I))
    cnt = await tools_btn.count()

    if cnt == 0:
        tools_btn = page.locator('button:has-text("Tools"), [role="button"]:has-text("Tools")')
        cnt = await tools_btn.count()

    if cnt > 0:
        out["tools_button_found"] = True
        btn0 = tools_btn.first

        try:
            out["tools_button_disabled"] = await btn0.is_disabled()
        except Exception:
            try:
                attrs = await btn0.evaluate(
                    """(el) => ({
                        disabled: el.hasAttribute('disabled'),
                        ariaDisabled: el.getAttribute('aria-disabled'),
                        dataDisabled: el.getAttribute('data-disabled'),
                        dataState: el.getAttribute('data-state'),
                    })"""
                )
                out["tools_button_disabled"] = bool(
                    attrs.get("disabled")
                    or str(attrs.get("ariaDisabled") or "").lower() == "true"
                    or attrs.get("dataDisabled") is not None
                    or str(attrs.get("dataState") or "").lower() == "disabled"
                )
            except Exception:
                out["tools_button_disabled"] = None

        try:
            out["tools_button_outer_html"] = await btn0.evaluate("(el) => el.outerHTML")
        except Exception:
            out["tools_button_outer_html"] = None

    # Wrench icon (post-JS)
    wrench = page.locator(
        'svg[data-icon-name="wrench"], '
        'svg[data-src*="wrench.svg"], '
        'use[href^="#wrench_"], '
        'symbol[id^="wrench_"]'
    )
    out["wrench_count"] = await wrench.count()
    out["wrench_found"] = out["wrench_count"] > 0

    return out

async def main(urls: List[str]):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        ctx = await browser.new_context(
            viewport={"width": 1400, "height": 900},
            user_agent=(
                "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
            ),
        )
        page = await ctx.new_page()

        results = []
        for url in urls:
            print("=" * 120)
            res = await detect_one(page, url)
            results.append(res)
            print("URL:", res["url"])
            print("TOOLS FOUND:", res["tools_button_found"])
            print("TOOLS DISABLED:", res["tools_button_disabled"])
            print("WRENCH FOUND:", res["wrench_found"], "COUNT:", res["wrench_count"])
            if res["tools_button_outer_html"]:
                print("\nTOOLS OUTER HTML (first 600 chars):")
                print(res["tools_button_outer_html"][:600])

        await ctx.close()
        await browser.close()
        return results


In [4]:
# Run (Jupyter-friendly)
results = await main(URLS)
results


URL: https://build.nvidia.com/openai/gpt-oss-120b
TOOLS FOUND: True
TOOLS DISABLED: False
WRENCH FOUND: True COUNT: 3

TOOLS OUTER HTML (first 600 chars):
<button aria-expanded="false" data-testid="nv-side-panel-trigger" type="button" aria-haspopup="dialog" aria-controls="radix-:rf:" data-state="closed" class="inline-flex items-center justify-center gap-2 text-center font-sans font-bold leading-text flex-row btn-secondary btn-sm btn-rounded btn-inverse"><svg data-src="https://brand-assets.cne.ngc.nvidia.com/assets/icons/3.8.0/fill/wrench.svg" height="1em" width="1em" display="inline-block" data-icon-name="wrench" data-cache="disabled" class="btn-icon" fill="none" viewBox="0 0 16 16" xmlns="http://www.w3.org/2000/svg" data-id="svg-loader_5"><symb
URL: https://build.nvidia.com/deepseek-ai/deepseek-r1
TOOLS FOUND: False
TOOLS DISABLED: None
WRENCH FOUND: False COUNT: 0
URL: https://build.nvidia.com/deepseek-ai/deepseek-r1-0528
TOOLS FOUND: True
TOOLS DISABLED: False
WRENCH FOUND: True COU

[{'url': 'https://build.nvidia.com/openai/gpt-oss-120b',
  'tools_button_found': True,
  'tools_button_disabled': False,
  'tools_button_outer_html': '<button aria-expanded="false" data-testid="nv-side-panel-trigger" type="button" aria-haspopup="dialog" aria-controls="radix-:rf:" data-state="closed" class="inline-flex items-center justify-center gap-2 text-center font-sans font-bold leading-text flex-row btn-secondary btn-sm btn-rounded btn-inverse"><svg data-src="https://brand-assets.cne.ngc.nvidia.com/assets/icons/3.8.0/fill/wrench.svg" height="1em" width="1em" display="inline-block" data-icon-name="wrench" data-cache="disabled" class="btn-icon" fill="none" viewBox="0 0 16 16" xmlns="http://www.w3.org/2000/svg" data-id="svg-loader_5"><symbol id="wrench_6" viewBox="0 0 16 16"><path fill="currentColor" d="M11,1c0.19333,0 0.38333,0.01333 0.57,0.04l0.43,0.062v0.398l-2.028,2.028l0.528,1.972l1.972,0.528l2.028,-2.028h0.398l0.062,0.43c0.02667,0.18667 0.04,0.37667 0.04,0.57c-0.00002,1.26085 -